In [24]:
import xarray as xr
import os
import pandas as pd

Juntando todos os arquivo .nc do conjunto de dados era5 e era5 land


In [25]:


# Caminho para a pasta onde estão os arquivos .nc
arquivos_nc_land = sorted([os.path.join(pasta_era5_land, f) for f in os.listdir(pasta_era5_land) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5_land = xr.open_mfdataset(arquivos_nc_land, combine='by_coords')

# Caminho para a pasta onde estão os arquivos .nc
arquivos_nc = sorted([os.path.join(pasta_era5, f) for f in os.listdir(pasta_era5) if f.endswith(".nc")])

# Abrir e concatenar ao longo da dimensão "time" (ou outra, se necessário)
ds_era5 = xr.open_mfdataset(arquivos_nc, combine='by_coords')


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 land

In [26]:


# Coordenadas da estação
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir o arquivo NetCDF
ds = ds_era5_land

# Verifique os nomes corretos das dimensões
print(ds.dims)
print(ds.coords)

# Substituir nomes de coordenadas se necessário
if 'lat' in ds.coords and 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Encontrar os índices mais próximos da coordenada desejada
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Selecionar os dados no ponto mais próximo
ponto = ds.isel(latitude=lat_idx, longitude=lon_idx)

# Usar 'valid_time' como eixo temporal
tempo = ds['valid_time'].values

# Extrair dados para todas as variáveis dependentes de 'valid_time'
dados = {}
for var in ds.data_vars:
    dims = ds[var].dims
    if 'valid_time' in dims:
        dados[var] = ponto[var].values
    else:
        # Repete o valor único ao longo do tempo
        dados[var] = [ponto[var].values] * len(tempo)

# Criar DataFrame
df = pd.DataFrame(dados)
df["valid_time"] = tempo
ds_era5_land_csv = df[["valid_time"] + [v for v in dados if v != "valid_time"]]



FrozenMappingWarningOnValuesAccess({'valid_time': 118344, 'latitude': 6, 'longitude': 4})
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 947kB 1940-01-01 ... 2020-12-31T1...
  * latitude    (latitude) float64 48B -8.65 -8.9 -9.15 -9.4 -9.65 -9.9
  * longitude   (longitude) float64 32B -77.9 -77.65 -77.4 -77.15
    number      int64 8B 0
    expver      (valid_time) object 947kB dask.array<chunksize=(1464,), meta=np.ndarray>


Criando um arquivo csv para a localização da estação Cuchillacocha para o arquivo era5 

In [27]:
import xarray as xr
import pandas as pd

# Coordenadas do ponto de interesse (Cuchillacocha)
lat_estacao = -9.41
lon_estacao = -77.35

# Abrir dataset
ds = ds_era5

# Renomear lat/lon se necessário
if 'lat' in ds.coords or 'lon' in ds.coords:
    ds = ds.rename({'lat': 'latitude', 'lon': 'longitude'})

# Achar índice do ponto mais próximo
lat_idx = abs(ds['latitude'] - lat_estacao).argmin()
lon_idx = abs(ds['longitude'] - lon_estacao).argmin()

# Verifica se existe a dimensão pressure_level
if "pressure_level" not in ds.dims:
    raise ValueError("O dataset não possui a dimensão 'pressure_level'.")

# Criar dicionário com DataFrames por nível de pressão
dfs_por_pressao = {}

for nivel in ds.pressure_level.values:
    ponto_nivel = ds.isel(latitude=lat_idx, longitude=lon_idx).sel(pressure_level=nivel)
    tempo = ds["valid_time"].values
    dados = {}

    for var in ds.data_vars:
        da = ponto_nivel[var]
        if "valid_time" in da.dims:
            # Reduz outras dimensões, se houver
            dims_extras = [d for d in da.dims if d != "valid_time"]
            if dims_extras:
                da = da.isel({d: 0 for d in dims_extras})
            dados[var] = da.values
        else:
            dados[var] = [da.values.item()] * len(tempo)

    df = pd.DataFrame(dados)
    df["valid_time"] = tempo
    df = df[["valid_time"] + [v for v in dados if v != "valid_time"]]

    dfs_por_pressao[int(nivel)] = df

# Empilhar todos os DataFrames com coluna 'pressure_level'
lista_df = []

for nivel, df in dfs_por_pressao.items():
    df_com_nivel = df.copy()
    df_com_nivel["pressure_level"] = nivel
    lista_df.append(df_com_nivel)

df_empilhado = pd.concat(lista_df, ignore_index=True)

# Reorganizar colunas (opcional)
colunas_ordenadas = ["valid_time", "pressure_level"] + [col for col in df_empilhado.columns if col not in ["valid_time", "pressure_level"]]
df_era5_csv = df_empilhado[colunas_ordenadas]

# ✅ df_empilhado agora contém todos os dados organizados por tempo e nível de pressão
print("✅ DataFrame final criado com sucesso!")

# Exemplo de uso:
# print(df_empilhado.head())


✅ DataFrame final criado com sucesso!


In [28]:
df_era5_csv

,valid_time,pressure_level,z,q,crwc,t,u,v
0,1940-01-01 00:00:00,500,57489.031250,0.002608,0.0,268.483124,0.371095,6.677262
1,1940-01-01 01:00:00,500,57528.335938,0.002191,0.0,268.860870,0.057231,6.478654
2,1940-01-01 02:00:00,500,57564.855469,0.001929,0.0,269.163086,-0.361216,6.226668
3,1940-01-01 03:00:00,500,57622.351562,0.001793,0.0,269.276062,-1.235389,5.597896
4,1940-01-01 04:00:00,500,57657.062500,0.001728,0.0,269.255127,-1.945148,4.694727
...,...,...,...,...,...,...,...,...
2125795,2020-11-15 19:00:00,300,95100.453125,0.000348,0.0,241.747437,-0.385240,1.356631
2125796,2020-11-15 20:00:00,300,95067.328125,0.000429,0.0,241.928040,-1.312189,1.560719
2125797,2020-11-15 21:00:00,300,95036.671875,0.000498,0.0,242.342667,-1.452204,2.107743
2125798,2020-11-15 22:00:00,300,95046.625000,0.000434,0.0,242.087677,-1.116784,4.098901


In [38]:
ds_era5_land_csv_sem_nan = ds_era5_land_csv.dropna()  # Remove colunas com todos os valores NaN
df_era5_csv_sem_nan = df_era5_csv.dropna()  # Remove colunas com todos os valores NaN

Fazendo a média e acumulação para cada variável era 5 land

In [36]:
import pandas as pd

# Copiar o DataFrame original
df = ds_era5_land_csv_sem_nan.copy()

# Converter coluna de tempo (se necessário)
df["valid_time"] = pd.to_datetime(df["valid_time"])
df["date"] = df["valid_time"].dt.date

# Listas de variáveis
variaveis_instantaneas = ["u10", "v10", "d2m", "t2m", "sp", "z"]
variaveis_acumuladas   = ["tp", "ssrd", "strd", "sf"]

# Média das variáveis instantâneas
df_instantaneas = df.groupby("date")[variaveis_instantaneas].mean().reset_index()

# Soma das acumuladas
df_acumuladas = df.groupby("date")[variaveis_acumuladas].sum().reset_index()

# Conversões:
# - tp → mm
df_acumuladas["tp"] = df_acumuladas["tp"] * 1000

# - t2m → °C
df_instantaneas["t2m"] = df_instantaneas["t2m"] - 273.15

# (Opcional: d2m também costuma ser em Kelvin → podemos converter para °C também se quiser!)
df_instantaneas["d2m"] = df_instantaneas["d2m"] - 273.15

# Juntar tudo
df_era_5_land_completo = pd.merge(df_instantaneas, df_acumuladas, on="date")

# Exemplo de visualização
df_era_5_land_completo


,date,u10,v10,d2m,t2m,sp,z,tp,ssrd,strd,sf
0,1940-01-01,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
1,1940-01-02,0.588264,0.229326,1.566925,4.754242,61876.953125,40786.863281,0.586233,3717622.75,4054675.25,1.192129e-07
2,1940-01-03,0.310574,-0.197998,1.243683,4.603973,61876.234375,40786.863281,0.573084,3139294.75,4116405.50,1.311237e-07
3,1940-01-04,0.193879,-0.018040,2.397186,4.376068,61869.957031,40786.863281,1.803716,2481867.75,4220082.00,2.153246e-04
4,1940-01-05,0.398864,0.188723,2.149017,3.874969,61843.035156,40786.863281,0.953280,2525109.00,4373661.00,1.110633e-04
...,...,...,...,...,...,...,...,...,...,...,...
29216,2020-12-27,0.847211,0.476750,2.063751,3.631989,61651.832031,40786.863281,1.770064,1614066.00,4483425.00,4.876889e-04
29217,2020-12-28,0.592481,-0.031748,1.099762,2.792389,61778.027344,40786.863281,1.755811,3226387.25,4426724.00,7.736497e-04
29218,2020-12-29,0.754008,0.241566,2.066620,4.761810,61838.539062,40786.863281,1.052107,3301645.50,4403458.50,7.603457e-05
29219,2020-12-30,0.636928,0.491266,1.414825,4.944672,61817.257812,40786.863281,0.267223,3391407.00,4252856.00,1.072767e-07


Fazendo a média para cada variável era 5

In [40]:
import pandas as pd

# Copiar o DataFrame
df = df_era5_csv_sem_nan.copy()

# Converter coluna de tempo (se necessário)
df["valid_time"] = pd.to_datetime(df["valid_time"])
df["date"] = df["valid_time"].dt.date

# Lista de variáveis instantâneas
variaveis_instantaneas = ["z", "t", "crwc", "q", "u", "v"]

# Agrupar por data + nível de pressão, tirar média
df_era_5_completo = df.groupby(["date", "pressure_level"])[variaveis_instantaneas].mean().reset_index()

# Conversão opcional:
# - t (temperatura) K → °C
df_era_5_completo["t"] = df_era_5_completo["t"] - 273.15

# Exemplo de visualização
df_era_5_completo

,date,pressure_level,z,t,crwc,q,u,v
0,1940-01-01,300,94812.320312,-33.647003,0.000000e+00,0.000446,-6.099928,-3.461634
1,1940-01-01,400,74360.429688,-17.487457,0.000000e+00,0.001279,0.045611,1.453575
2,1940-01-01,500,57534.500000,-4.616638,0.000000e+00,0.002186,-0.382076,3.116183
3,1940-01-02,300,94770.718750,-33.238312,0.000000e+00,0.000507,-2.169100,0.126608
4,1940-01-02,400,74318.554688,-17.626190,0.000000e+00,0.001330,4.247507,2.603156
...,...,...,...,...,...,...,...,...
88570,2020-11-14,400,74444.460938,-15.358704,0.000000e+00,0.001001,-3.696155,4.267547
88571,2020-11-14,500,57569.199219,-4.821045,2.080393e-09,0.003882,-2.040295,1.537333
88572,2020-11-15,300,95082.054688,-31.506927,0.000000e+00,0.000294,1.188069,2.133075
88573,2020-11-15,400,74436.906250,-15.169556,0.000000e+00,0.000904,-7.737541,3.062083


Sera necessário ? 

In [51]:
# Primeiro, criar a coluna pressure_level no land para poder juntar
df_era_5_land_completo["pressure_level"] = None  # ou np.nan

# Agora, fazer um merge com um outer join pelo campo 'date' e 'pressure_level'
df_final = pd.merge(
    df_era_5_completo,
    df_era_5_land_completo,
    on="date",
    how="left"  # mantém todos os níveis de pressão
)

# Remove apenas a coluna pressure_level_y
df_final = df_final.drop(columns=["pressure_level_y"])


df_final


,date,pressure_level_x,z_x,t,crwc,q,u,v,u10,v10,d2m,t2m,sp,z_y,tp,ssrd,strd,sf
0,1940-01-01,300,94812.320312,-33.647003,0.000000e+00,0.000446,-6.099928,-3.461634,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
1,1940-01-01,400,74360.429688,-17.487457,0.000000e+00,0.001279,0.045611,1.453575,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
2,1940-01-01,500,57534.500000,-4.616638,0.000000e+00,0.002186,-0.382076,3.116183,0.328639,0.214869,-0.051605,6.205353,61904.390625,40786.863281,0.029278,3277854.50,1974766.25,0.000000e+00
3,1940-01-02,300,94770.718750,-33.238312,0.000000e+00,0.000507,-2.169100,0.126608,0.588264,0.229326,1.566925,4.754242,61876.953125,40786.863281,0.586233,3717622.75,4054675.25,1.192129e-07
4,1940-01-02,400,74318.554688,-17.626190,0.000000e+00,0.001330,4.247507,2.603156,0.588264,0.229326,1.566925,4.754242,61876.953125,40786.863281,0.586233,3717622.75,4054675.25,1.192129e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88570,2020-11-14,400,74444.460938,-15.358704,0.000000e+00,0.001001,-3.696155,4.267547,0.207066,0.205525,4.106750,6.518219,61897.375000,40786.863281,1.566299,2828654.00,4501893.50,3.185589e-05
88571,2020-11-14,500,57569.199219,-4.821045,2.080393e-09,0.003882,-2.040295,1.537333,0.207066,0.205525,4.106750,6.518219,61897.375000,40786.863281,1.566299,2828654.00,4501893.50,3.185589e-05
88572,2020-11-15,300,95082.054688,-31.506927,0.000000e+00,0.000294,1.188069,2.133075,-0.260562,-0.226027,4.477448,6.869965,61885.839844,40786.863281,1.312168,3891015.50,4224658.00,2.090156e-04
88573,2020-11-15,400,74436.906250,-15.169556,0.000000e+00,0.000904,-7.737541,3.062083,-0.260562,-0.226027,4.477448,6.869965,61885.839844,40786.863281,1.312168,3891015.50,4224658.00,2.090156e-04
